# Compare genomes: four E. coli strains in a linear synteny view

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GMOD/jbrowse-anywidget/blob/main/examples/11_synteny_ecoli.ipynb)

`JBrowseApp` drives the full app, so a `views=[...]` list can hold a `LinearSyntenyView` — several genomes stacked, the blocks each pair shares drawn between the rows. Here are four *E. coli* strains (K12, Sakai, CFT073, NCTC86) tied together by one all-vs-all minimap2 alignment, the same data as the [all-vs-all synteny tutorial](https://jbrowse.org/jb2/docs/tutorials/allvsall_synteny/). Everything below is hosted, so this cell runs as-is.

In [ ]:
# Install only if not already available (e.g. in Colab). The GitHub install
# needs no JS toolchain — the built widget bundle is committed in the repo. A
# local editable install is used as-is. (Swap to `jbrowse-anywidget` once it's
# published to PyPI.)
try:
    import jbrowse_anywidget  # noqa: F401
except ImportError:
    %pip install -q "jbrowse-anywidget @ git+https://github.com/GMOD/jbrowse-anywidget" pandas numpy

# Colab requires this to render third-party (anywidget) widgets:
try:
    from google.colab import output

    output.enable_custom_widget_manager()
except ImportError:
    pass

## Stack the four strains, one all-vs-all track between them

Each genome is a `make_assembly` from its hosted FASTA. The single `AllVsAllPAFAdapter` track serves every pair from one PAF, so the three bands between the four rows are all the same trackId (`tracks=[["ecoli_ava"]] * 3`, one entry per adjacent pair). `drawCurves=False` draws straight ribbons; `minAlignmentLength` hides short noisy blocks.

In [ ]:
from jbrowse_anywidget import JBrowseApp, make_assembly, synteny_view

BASE = "https://jbrowse.org/demos/ecoli_pangenome"
STRAINS = ["K12", "Sakai", "CFT073", "NCTC86"]

assemblies = [make_assembly(s, f"{BASE}/{s}.fa.gz") for s in STRAINS]

ecoli_ava = {
    "type": "SyntenyTrack",
    "trackId": "ecoli_ava",
    "name": "E. coli all-vs-all (minimap2 PAF)",
    "assemblyNames": STRAINS,
    "adapter": {
        "type": "AllVsAllPAFAdapter",
        "assemblyNames": STRAINS,
        "pafLocation": {"uri": f"{BASE}/all_vs_all.paf.gz"},
    },
}

JBrowseApp(
    assemblies=assemblies,
    tracks=[ecoli_ava],
    views=[
        synteny_view(
            STRAINS,
            tracks=[["ecoli_ava"]] * 3,  # one band per adjacent pair
            drawCurves=False,
            minAlignmentLength=10000,
        )
    ],
)

The same PAF also opens as a **dotplot** — swap `synteny_view` for `dotplot_view(["K12", "Sakai"], tracks=["ecoli_ava"])` to see any one pair whole-genome. To build the PAF from your own genomes (`minimap2 -c -x asm20 --eqx`) and load per-strain gene tracks alongside, follow the [tutorial](https://jbrowse.org/jb2/docs/tutorials/allvsall_synteny/).